<a href="https://colab.research.google.com/github/Hanna07111/masked-social-signals/blob/review-notes/VQVAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### encoding
- vqvae 입력: 1 세그먼트(30초, 90프레임) 1모달 1사람 => (Batch, Feature, 90)
- 배치 입력: (bzx3x12, Feature, 90)
1. CNN encoding (bzx3x12, h_dim, 90) -> h_dim -> 32로 바로 정해주지 않은 이유?? 그렇게 하면 2번 과정 거치지 않아도 됐을텐디 => 한 번에 저차원으로 축소하면 표현력이 떨어질 수 있음
2. Conv1d => prequantization (bzx3x12, e_dim, 90)
3. Quantization (bzx3x12, e_dim, 90)
4. flatten (bzx3x12, Feature*Length)
5. linear projection (1152, 1024)

In [ ]:
def encode(self, x):
        # CNN Encoder 차원 맞추기
        x = x.permute(0, 2, 1).contiguous()
        # CNN Encoder -> (Batch, h_dim, Length)
        z_e = self.encoder(x)

        # Conv1d로 pre_quantization -> (Batch, 32, Length) (kernel은 128x1 크기로 32개)
        # 여기서 kernel size => time step을 몇 개 참고할건지
        z_e = self.pre_quantization_conv(z_e) #.permute(0, 2, 1).contiguous()

        # Quantization -> (Batch, 32, Length)
        embedding_loss, z_q, perplexity = self.vector_quantization(z_e)
        self.hidden_shape = z_q.shape

        # 한 사람, 한 세그먼트 당의 내용이 flatten
        # 즉, 사람1, 세그먼트1의 정보 -> 행렬 => 하나의 토큰 벡터
        # (bz*3*12, Feature*Length)
        z_q = z_q.flatten(start_dim=1) # (1152=bz*3*12, 32*90) -> (1152, 1024) -> transformer -> (1152, 1024)

        #  선형 변환 -> 차원 맞춰주기 위함
        linear_proj = self.linear_projector.encode(z_q)

        return embedding_loss, linear_proj, perplexity

### decode

- 인코딩과는 완전히 반대 과정

1. 선형 변환 (디코딩) (bzx3x12, Feature*Length)
2. 1의 결과를 -> (bzx3x12, e_dim, Length) <-로 reshape
3. Quantization (bzx3x12, e_dim, Length)
- 인코더와 디코더 간 일관성을 유지하고, codebook 학습 안정화를 위해
4. CNN decoding -> (bzx3x12, Feature, Length)

In [ ]:
def decode(self, z, hard=True): # (bz, 3, 12, 1024)

        # 선형 변환 -> 디코딩
        linear_proj = self.linear_projector.decode(z)

        # self.hidden_shape = (bzx3x12, e_dim, 90)
        z_reshaped = linear_proj.view(self.hidden_shape)

        # Quantization
        embedding_loss, z_e, perplexity = self.vector_quantization(z_reshaped, hard=hard)

        # CNN 디코더로 디코딩
        # import pdb; pdb.set_trace()
        x_hat = self.decoder(z_e).permute(0, 2, 1).contiguous()
        return embedding_loss, x_hat, perplexity

### Quantization
1. codebook 가중치 초기화
2. 입력값 flatten (Batch*length, e_dim)
3. codebook과 거리계산 및 토큰화
4. z_q 구하기 -> 차원은 (Batch, e_dim, Length)로 복원
5. compute embedding loss (beta 사용)
6. loss, z_q, perplexity 반환

In [ ]:
# Quantization
class VectorQuantizer(nn.Module):

    def __init__(self, n_e, e_dim, beta):
        super(VectorQuantizer, self).__init__()

        # codebook 벡터 개수
        self.n_e = n_e # 512

        # codebook 벡터 차원
        self.e_dim = e_dim # 64

        # commitment loss 가중치
        self.beta = beta

        # codebook 생성
        self.embedding = nn.Embedding(self.n_e, self.e_dim)

        # codebook 가중치 초기화
        self.embedding.weight.data.uniform_(-1.0 / self.n_e, 1.0 / self.n_e)

    def forward(self, z, hard=True):
        # reshape z -> (batch, height, width, channel) and flatten
        # (Batch, 32, Length) => (Batch*Length, 32)
        # 32개 차원(특징)을 가진 벡터가 현재 Batch*Length개 있음 => 이것들을 모두 한 평면에 평면화
        z_flattened = z.view(-1, self.e_dim)

        # codebook과 거리계산
        # distances from z to embeddings e_j (z - e)^2 = z^2 + e^2 - 2 e * z
        d = torch.sum(z_flattened ** 2, dim=1, keepdim=True) + \
            torch.sum(self.embedding.weight**2, dim=1) - 2 * \
            torch.matmul(z_flattened, self.embedding.weight.t())

        # find closest encodings
        if hard:
            min_encoding_indices = torch.argmin(d, dim=1).unsqueeze(1)
            min_encodings = torch.zeros(
                min_encoding_indices.shape[0], self.n_e).to(z.device)
            min_encodings.scatter_(1, min_encoding_indices, 1)

            # get quantized latent vectors
            z_q = torch.matmul(min_encodings, self.embedding.weight).view(z.shape)

        else:
            # soft selection
            softmax_d = F.softmax(-d, dim=1)
            z_q = torch.matmul(softmax_d, self.embedding.weight).view(z.shape)

        # compute loss for embedding
        loss = torch.mean((z_q.detach()-z)**2) + self.beta * \
            torch.mean((z_q - z.detach()) ** 2)


        if hard:
            # preserve gradients
            z_q = z + (z_q - z).detach()

            # perplexity -> 코드북의 사용빈도분포를 계산, 값이 클수록 코드북을 골고루 사용하고 있다는 뜻
            e_mean = torch.mean(min_encodings, dim=0)
            perplexity = torch.exp(-torch.sum(e_mean * torch.log(e_mean + 1e-10)))

        else:
            perplexity = None

        return loss, z_q, perplexity # min_encodings, min_encoding_indices